# Explore a near-optimal set

This notebook explores the near-optimal designs of PyPSA's `model_energy` example: a
single-node system with wind, solar, battery and hydrogen storage. It starts from the
cheapest design, asks for more wind power, looks at the cost of getting there, and at what
changes in the system on the way.

It needs a **near-optimal set** of the model. Build one with the dataset script (about 10
minutes; see the *Dataset scripts* reference), from the `mga-compass` directory:

```bash
uv run --extra scripts python scripts/generate_dataset.py --tol 5 --max-iter 120 --max-parallel 4
```

and run this notebook with the `notebook` extra, for plots:

```bash
uv run --extra notebook --with jupyter jupyter lab
```

In [ ]:
import pandas as pd

import mga_compass as mc

pd.set_option("display.float_format", "{:,.1f}".format)

SET = "../scripts/output/set"  # the set generate_dataset.py writes; or its .zip, or a URL

## Load the set

`mc.load` reads the set and checks it. The result is a `Space`: the set's designs in the
space of its properties of interest (PoIs), here the capacity of each technology, plus their
cost. Values are in the set's display units (the cost in M€). By default, the space is the
largest slack level of the set.

In [ ]:
space = mc.load(SET)
space

In [ ]:
space.points.head()

## Ask for more wind

Start from the cheapest design and ask for 20% more wind capacity, while the cost may rise by
at most 3%. The order of the requests is their priority: wind first, then cost. Solar,
battery and hydrogen are not listed, so they are free to change however is cheapest.

In [ ]:
result = space.navigate(space.optimum, {
    "wind": ("increase", "20%"),
    "TOTEX": ("keep", "3%"),
})
result

If a request cannot be met in full, it is met as far as possible, and the result says
how it was adjusted (`result.changes`).

## The way there

`result.traverse` is the way from the start to the target: the PoIs move in a straight line,
and the cost along the way is the cheapest mix of the set's designs. The designs' duals give
a lower bound: the true cost of the designs on the way lies between the two.

In [ ]:
result.traverse

In [ ]:
result.traverse.plot();

## What changes in the system

Every design here is a mix of the set's designs, and its detailed outputs are the same mix of
theirs. `space.outputs()` lists the outputs; `space.output` gives one for a design: a
navigation result (its target), the weights of any point along the traverse, or a design of
the set by its id.

In [ ]:
space.outputs()

In [ ]:
pd.DataFrame({
    "start": space.output("capacity", result.traverse.alpha_at(0)),
    "halfway": space.output("capacity", result.traverse.alpha_at(0.5)),
    "target": space.output("capacity", result),
})

Outputs over time come with a `snapshot` level; here the target's generation, as weekly
means:

In [ ]:
generation = space.output("generation", result).unstack()
generation.resample("W").mean().plot.area(ylabel="generation [MW], weekly mean", linewidth=0);

## Carry on

Navigation can start from any result. From the design with more wind, ask for at least 50%
more hydrogen storage, keeping wind where it is and the cost within 2% of the start's.

After meeting every request, navigation improves them one by one, in their order: an
increase goes as far as the requests before it allow, a keep stays as close as they allow.
With hydrogen before the cost, hydrogen grows until the cost reaches its 2% limit; with the
cost first, the cost would stay as close to the start's as possible, and hydrogen would grow
by just the 50%.

In [ ]:
space.navigate(result, {
    "wind": "keep",
    "hydrogen": ("increase", "50%"),
    "TOTEX": ("keep", "2%"),
})

## A tighter budget

`slack` picks a smaller slack level of the set: only the designs found within that much of
the optimal cost.

In [ ]:
tight = mc.load(SET, slack=0.02)
tight.navigate(tight.optimum, {"wind": ("increase", "20%")})